# Preprocessing Pipeline

Notebook này triển khai đầy đủ theo insight từ EDA:
- Làm sạch và xử lý missing theo cấu trúc Student/Working Professional
- Tạo đặc trưng mới (feature engineering)
- Binning cho các biến liên tục
- Encoding cho biến phân loại
- Scaling cho biến số

Đầu ra:
- `train_preprocessed.csv`
- `test_preprocessed.csv`

## 0) Nạp thư viện và đọc dữ liệu gốc

**Mục tiêu**
- Import các thư viện cần dùng cho preprocessing: `pandas`, `numpy`, `StandardScaler`.
- Đọc `train.csv` và `test.csv` vào DataFrame.
- In shape và vài dòng đầu để xác nhận dữ liệu vào đúng định dạng.

**Logic xử lý**
- Chưa biến đổi dữ liệu ở bước này.
- Chỉ thiết lập môi trường và kiểm tra nhanh đầu vào trước khi đi vào các bước làm sạch/biến đổi.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(train.head(2))

Train shape: (140700, 20)
Test shape : (93800, 19)
   id      Name  Gender   Age      City Working Professional or Student  \
0   0  Aaradhya  Female  49.0  Ludhiana            Working Professional   
1   1     Vivan    Male  26.0  Varanasi            Working Professional   

  Profession  Academic Pressure  Work Pressure  CGPA  Study Satisfaction  \
0       Chef                NaN            5.0   NaN                 NaN   
1    Teacher                NaN            4.0   NaN                 NaN   

   Job Satisfaction     Sleep Duration Dietary Habits Degree  \
0               2.0  More than 8 hours        Healthy    BHM   
1               3.0  Less than 5 hours      Unhealthy    LLB   

  Have you ever had suicidal thoughts ?  Work/Study Hours  Financial Stress  \
0                                    No               1.0               2.0   
1                                   Yes               7.0               3.0   

  Family History of Mental Illness  Depression  
0             

## 1) Cleaning, xử lý missing và clip outlier theo insight từ EDA

**Mục tiêu**
- Chuẩn hóa các giá trị phân loại bẩn (`Dietary Habits`, `Sleep Duration`, `Degree`).
- Điền missing theo đúng bản chất dữ liệu (structural missing và random missing).
- Clip outlier theo domain rule (`Age`, `Work/Study Hours`).

**Logic xử lý chính**
- Tạo các hàm làm sạch để gom giá trị về nhóm chuẩn, áp dụng đồng nhất cho cả train/test.
- Với nhóm Working Professional: cột học tập (`Academic Pressure`, `Study Satisfaction`, `CGPA`) điền `0` vì "không áp dụng".
- Với nhóm Student: cột công việc (`Work Pressure`, `Job Satisfaction`) điền `0` vì "không áp dụng".
- Missing nhỏ lẻ: dùng thống kê từ **train** (mode/median) rồi map sang test để tránh leakage.
- Kiểm tra tổng số missing còn lại sau xử lý.

In [2]:
# -----------------------------
# 1) Cleaning + Missing + Outlier clipping
# -----------------------------

def clean_dietary(val):
    val = str(val).strip()
    diet_map = {
        "Healthy": "Healthy",
        "More Healthy": "Healthy",
        "Moderate": "Moderate",
        "Unhealthy": "Unhealthy",
        "No Healthy": "Unhealthy",
        "Yes": "Moderate",
        "No": "Moderate",
        "Class 12": "Moderate",
    }
    return diet_map.get(val, "Moderate")


def clean_sleep(val):
    val = str(val).lower().strip()
    if any(x in val for x in ["less than 5", "2-3", "3-4", "4-5"]):
        return "Less than 5 hours"
    if "5-6" in val:
        return "5-6 hours"
    if any(x in val for x in ["6-7", "7-8"]):
        return "7-8 hours"
    if "more than 8" in val or "8" in val:
        return "More than 8 hours"
    return "7-8 hours"


def group_degree(val):
    val = str(val).strip()
    if val in ["Class 12", "High School"]:
        return "High School"
    if val.startswith("B.") or (val.startswith("B") and len(val) <= 5) or val in ["BCA", "BBA", "BHM"]:
        return "Bachelor"
    if val.startswith("M.") or (val.startswith("M") and len(val) <= 5) or val in ["MCA", "MBA", "MHM"]:
        return "Master"
    if val.startswith("Ph") or ("PhD" in val) or ("Phd" in val):
        return "PhD"
    return "Bachelor"


def basic_clean(df):
    df = df.copy()

    # Categorical normalization from EDA
    df["Dietary Habits"] = df["Dietary Habits"].apply(clean_dietary)
    df["Sleep Duration"] = df["Sleep Duration"].apply(clean_sleep)
    df["Degree"] = df["Degree"].apply(group_degree)
    df["Profession"] = df["Profession"].fillna("Student")

    # Outlier clipping with domain rules from EDA
    df["Age"] = df["Age"].clip(15, 80)
    df["Work/Study Hours"] = df["Work/Study Hours"].clip(0, 16)

    return df


train = basic_clean(train)
test = basic_clean(test)

# Structural missing
academic_cols = ["Academic Pressure", "Study Satisfaction", "CGPA"]
work_cols = ["Work Pressure", "Job Satisfaction"]

for df in [train, test]:
    wp_mask = df["Working Professional or Student"] == "Working Professional"
    st_mask = df["Working Professional or Student"] == "Student"

    df.loc[wp_mask, academic_cols] = df.loc[wp_mask, academic_cols].fillna(0)
    df.loc[st_mask, work_cols] = df.loc[st_mask, work_cols].fillna(0)

# Random missing (fit from train)
for col in ["Dietary Habits", "Degree"]:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(train[col].mode()[0])

train["Financial Stress"] = train["Financial Stress"].fillna(train["Financial Stress"].median())
test["Financial Stress"] = test["Financial Stress"].fillna(train["Financial Stress"].median())

# Any residual missing in structural columns
residual_structural = ["Academic Pressure", "CGPA", "Study Satisfaction", "Work Pressure", "Job Satisfaction"]
for col in residual_structural:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

print("Missing after cleaning - train:", int(train.isna().sum().sum()))
print("Missing after cleaning - test :", int(test.isna().sum().sum()))

Missing after cleaning - train: 0
Missing after cleaning - test : 0


## 2) Feature Engineering: tạo đặc trưng mới từ tín hiệu stress, hài lòng và lối sống

**Mục tiêu**
- Tạo các feature tổng hợp giúp model nhìn rõ hơn quan hệ giữa áp lực, mức hài lòng và giờ sinh hoạt.

**Logic xử lý chính**
- Tạo biến tổng hợp: `overall_pressure`, `overall_satisfaction`.
- Tạo biến chênh lệch: `stress_satisfaction_gap` để phản ánh mất cân bằng tâm lý.
- Quy đổi `Sleep Duration` thành số giờ ước lượng (`sleep_hours_est`) để tính được các tỉ lệ.
- Tạo đặc trưng tương tác: `work_sleep_ratio`, `stress_per_hour`, `age_x_workload`.
- Tạo cờ nhị phân vai trò (`is_working_professional`) để mô hình học khác biệt theo nhóm.

In [3]:
# -----------------------------
# 2) Feature engineering
# -----------------------------

def add_features(df):
    df = df.copy()

    # Composite stress signals
    df["overall_pressure"] = (df["Academic Pressure"] + df["Work Pressure"]) / 2.0
    df["overall_satisfaction"] = (df["Study Satisfaction"] + df["Job Satisfaction"]) / 2.0
    df["stress_satisfaction_gap"] = df["overall_pressure"] - df["overall_satisfaction"]

    # Lifestyle & workload interactions
    df["sleep_hours_est"] = df["Sleep Duration"].map(
        {
            "Less than 5 hours": 4.5,
            "5-6 hours": 5.5,
            "7-8 hours": 7.5,
            "More than 8 hours": 8.5,
        }
    )
    df["work_sleep_ratio"] = df["Work/Study Hours"] / df["sleep_hours_est"].replace(0, np.nan)
    df["stress_per_hour"] = df["Financial Stress"] / (df["Work/Study Hours"] + 1)

    # Nonlinear / grouped by role
    df["is_working_professional"] = (df["Working Professional or Student"] == "Working Professional").astype(int)
    df["age_x_workload"] = df["Age"] * df["Work/Study Hours"]

    return df


train = add_features(train)
test = add_features(test)

print("Added engineered features:")
new_feats = [
    "overall_pressure",
    "overall_satisfaction",
    "stress_satisfaction_gap",
    "sleep_hours_est",
    "work_sleep_ratio",
    "stress_per_hour",
    "is_working_professional",
    "age_x_workload",
]
print(new_feats)

Added engineered features:
['overall_pressure', 'overall_satisfaction', 'stress_satisfaction_gap', 'sleep_hours_est', 'work_sleep_ratio', 'stress_per_hour', 'is_working_professional', 'age_x_workload']


## 3) Binning: rời rạc hóa các biến liên tục thành nhóm mức độ

**Mục tiêu**
- Chuyển một số biến số thành các nhóm có ý nghĩa nghiệp vụ để tăng khả năng bắt pattern phi tuyến.

**Logic xử lý chính**
- `Age` được chia thành nhóm tuổi (`age_bin`).
- `Work/Study Hours` được chia thành mức tải công việc/học (`hours_bin`).
- `Financial Stress` được chia thành `low/medium/high` (`financial_stress_bin`).
- Dùng `pd.cut` với mốc cắt rõ ràng để đảm bảo tính giải thích và tái lập.

In [4]:
# -----------------------------
# 3) Binning
# -----------------------------

def add_binning(df):
    df = df.copy()

    df["age_bin"] = pd.cut(
        df["Age"],
        bins=[15, 24, 34, 44, 54, 80],
        labels=["15-24", "25-34", "35-44", "45-54", "55+"],
        include_lowest=True,
    )

    df["hours_bin"] = pd.cut(
        df["Work/Study Hours"],
        bins=[-0.1, 3, 6, 9, 12, 16],
        labels=["very_low", "low", "medium", "high", "very_high"],
    )

    df["financial_stress_bin"] = pd.cut(
        df["Financial Stress"],
        bins=[0, 2, 4, 5],
        labels=["low", "medium", "high"],
        include_lowest=True,
    )

    return df


train = add_binning(train)
test = add_binning(test)

print("Binning done:")
print(train[["age_bin", "hours_bin", "financial_stress_bin"]].head())

Binning done:
  age_bin hours_bin financial_stress_bin
0   45-54  very_low                  low
1   25-34    medium               medium
2   25-34  very_low                  low
3   15-24      high                  low
4   25-34    medium               medium


## 4) Encoding: chuyển toàn bộ biến phân loại về dạng số

**Mục tiêu**
- Biến đổi categorical features thành numeric features để model có thể học.

**Logic xử lý chính**
- Binary encoding cho các cột 2 giá trị (`Yes/No`, `Male/Female`, role).
- Ordinal encoding cho các cột có thứ tự tự nhiên (`Dietary Habits`, `Sleep Duration`, `Degree`) và các cột binning.
- Target encoding cho cột cardinality cao (`City`, `Profession`) bằng trung bình `Depression` trên train.
- Category mới ở test (unseen) được điền bằng global mean để ổn định.
- Loại bỏ cột `Name` vì là chuỗi tự do, ít giá trị dự báo trực tiếp.

In [5]:
# -----------------------------
# 4) Encoding
# -----------------------------

# Binary encoding
binary_map = {
    "Gender": {"Male": 1, "Female": 0},
    "Working Professional or Student": {"Working Professional": 1, "Student": 0},
    "Have you ever had suicidal thoughts ?": {"Yes": 1, "No": 0},
    "Family History of Mental Illness": {"Yes": 1, "No": 0},
}
for col, mapping in binary_map.items():
    train[col] = train[col].map(mapping)
    test[col] = test[col].map(mapping)

# Ordinal encoding
ordinal_map = {
    "Dietary Habits": {"Unhealthy": 0, "Moderate": 1, "Healthy": 2},
    "Sleep Duration": {
        "Less than 5 hours": 0,
        "5-6 hours": 1,
        "7-8 hours": 2,
        "More than 8 hours": 3,
    },
    "Degree": {"High School": 0, "Bachelor": 1, "Master": 2, "PhD": 3},
    "age_bin": {"15-24": 0, "25-34": 1, "35-44": 2, "45-54": 3, "55+": 4},
    "hours_bin": {"very_low": 0, "low": 1, "medium": 2, "high": 3, "very_high": 4},
    "financial_stress_bin": {"low": 0, "medium": 1, "high": 2},
}
for col, mapping in ordinal_map.items():
    train[col] = train[col].map(mapping)
    test[col] = test[col].map(mapping)

# Target encoding for high-cardinality variables (based on EDA)
for col in ["City", "Profession"]:
    target_mean = train.groupby(col)["Depression"].mean()
    global_mean = train["Depression"].mean()

    train[col] = train[col].map(target_mean)
    test[col] = test[col].map(target_mean).fillna(global_mean)

# Name is free text, usually not useful for modeling
drop_cols = ["Name"]
train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

print("Encoding completed.")
print("Remaining object columns (train):", train.select_dtypes("object").columns.tolist())
print("Remaining object columns (test) :", test.select_dtypes("object").columns.tolist())

Encoding completed.
Remaining object columns (train): []
Remaining object columns (test) : []


## 5) Scaling: chuẩn hóa các biến liên tục

**Mục tiêu**
- Đưa các continuous features về cùng thang đo để hỗ trợ các thuật toán nhạy cảm với scale.

**Logic xử lý chính**
- Chỉ scale các biến liên tục (không scale cột ID, target, hoặc các biến đã encode dạng mức/nhị phân).
- Fit `StandardScaler` trên train và transform cho cả train/test để tránh leakage.
- Giữ `id` để phục vụ join/submission ở bước downstream.

In [6]:
# -----------------------------
# 5) Scaling
# -----------------------------

# Keep id for downstream merging/submission
train_ids = train["id"].copy()
test_ids = test["id"].copy()

feature_cols = [c for c in train.columns if c not in ["Depression", "id"]]

# Scale continuous features only
continuous_cols = [
    "Age",
    "Work/Study Hours",
    "Academic Pressure",
    "Work Pressure",
    "CGPA",
    "Study Satisfaction",
    "Job Satisfaction",
    "Financial Stress",
    "overall_pressure",
    "overall_satisfaction",
    "stress_satisfaction_gap",
    "sleep_hours_est",
    "work_sleep_ratio",
    "stress_per_hour",
    "age_x_workload",
]
continuous_cols = [c for c in continuous_cols if c in feature_cols]

scaler = StandardScaler()
train[continuous_cols] = scaler.fit_transform(train[continuous_cols])
test[continuous_cols] = scaler.transform(test[continuous_cols])

print("Scaled columns:", continuous_cols)
print("Train shape after preprocessing:", train.shape)
print("Test shape after preprocessing :", test.shape)

Scaled columns: ['Age', 'Work/Study Hours', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Financial Stress', 'overall_pressure', 'overall_satisfaction', 'stress_satisfaction_gap', 'sleep_hours_est', 'work_sleep_ratio', 'stress_per_hour', 'age_x_workload']
Train shape after preprocessing: (140700, 30)
Test shape after preprocessing : (93800, 29)


## 6) Xuất dữ liệu đã preprocess để huấn luyện mô hình

**Mục tiêu**
- Lưu dữ liệu sau toàn bộ pipeline thành file CSV dùng cho training/inference.

**Logic xử lý chính**
- Tạo bản sao cuối cùng `train_preprocessed` và `test_preprocessed`.
- Ghi ra 2 file: `train_preprocessed.csv`, `test_preprocessed.csv`.
- In preview và danh sách cột feature để kiểm tra nhanh đầu ra.

In [7]:
# -----------------------------
# 6) Final output for modeling
# -----------------------------

train_preprocessed = train.copy()
test_preprocessed = test.copy()

train_preprocessed.to_csv("train_preprocessed.csv", index=False)
test_preprocessed.to_csv("test_preprocessed.csv", index=False)

print("Saved:")
print("- train_preprocessed.csv")
print("- test_preprocessed.csv")

print("\nTrain preview:")
print(train_preprocessed.head(3))

print("\nColumns used for training:")
print([c for c in train_preprocessed.columns if c not in ["id", "Depression"]])

Saved:
- train_preprocessed.csv
- test_preprocessed.csv

Train preview:
   id  Gender       Age      City  Working Professional or Student  \
0   0       0  0.695360  0.192690                                1   
1   1       1 -1.161867  0.142206                                1   
2   2       1 -0.596624  0.159196                                0   

   Profession  Academic Pressure  Work Pressure      CGPA  Study Satisfaction  \
0    0.048567          -0.446459       1.495218 -0.486345           -0.441965   
1    0.055649          -0.446459       0.919263 -0.486345           -0.441965   
2    0.534842           3.136517      -1.384554  2.386461            1.071869   

   Job Satisfaction  Sleep Duration  Dietary Habits  Degree  \
0         -0.221397               3               2       1   
1          0.354580               0               0       1   
2         -1.373352               1               2       1   

   Have you ever had suicidal thoughts ?  Work/Study Hours  Financial